# LUMA - Lokalny Urok Mistrzowstwa Artystycznego
LUMA to lokalny jarmark, który rośnie z tygodnia na tydzień: coraz więcej wystawców, więcej gości (miejscowych i turystów, okazjonalne promocje i sezonowość)

## Import bibliotek

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Wygenerowanie syntetycznych danych

### Ustawienie ziarna RNG

In [2]:
RNG_SEED = 20261020
rng = np.random.default_rng(RNG_SEED)
print("Ustawiono ziarno RNG: ", RNG_SEED)

Ustawiono ziarno RNG:  20261020


### Zdefiniowanie danych

In [3]:
LUMA_FULL = "Lokalny Urok Mistrzowstwa Artystycznego"
CATEGORIES = ["Ceramika", "Tekstylia", "Bizuteria", "Rękodzieła", "Przetwory", "Dekoracje", "Stolarka", "Masarstwo", "Alkochole"]

### Przygotowanie bazowych danych syntetycznych

"katalog" kategorii i ich bazowe parametry generatywne (syntetyczne)

In [4]:
catalog = pd.DataFrame({
    "category": CATEGORIES,
    "demand_loc_base": [128, 180, 130, 80, 160, 150, 70, 90, 110], # średni lokalny popyt (szt./tydz.)
    "demand_tur_base": [60, 40, 80, 30, 90, 30, 50, 40, 80], # średnie popyt turystyczny (szt./tydz.)
    "price_pin_base": [8, 120, 150, 230, 40, 35, 250, 215, 135] # średnia cena jednostkowa
})

Wyliczony "typowy" przychód tygodniowy: (lokalni + turyści)*cena

In [5]:
catalog["revenue_avg_pln"] = (catalog["demand_loc_base"]+catalog["demand_tur_base"])*catalog["price_pin_base"]

In [6]:
catalog

,category,demand_loc_base,demand_tur_base,price_pin_base,revenue_avg_pln
0,Ceramika,128,60,8,1504
1,Tekstylia,180,40,120,26400
2,Bizuteria,130,80,150,31500
3,Rękodzieła,80,30,230,25300
4,Przetwory,160,90,40,10000
5,Dekoracje,150,30,35,6300
6,Stolarka,70,50,250,30000
7,Masarstwo,90,40,215,27950
8,Alkochole,110,80,135,25650


### Przygotowanie funkcji pomomcniczych do generowania danych tygodniowych

Generowanie popytu lokalnego i turystycznego na podstawie danych bazowych, z losowym szumem (multiplikatywnym) i opcjonalnymi boostami (%)

In [7]:
def determine_demand(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    df = catalog[["category", "demand_loc)base", "demand_tur_base"]].copy()
    eps_loc = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    eps_tur = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    df["demand_loc"] = (df["demand_loc_base"] * (1 + boost_loc)) * eps_loc
    df["demand_tur"] = (df["demand_tur_base"] * (1 + boost_tur)) * eps_tur
    df["demand_loc"] = np.maximum(df["demand_loc"].round().astype(int), 0)
    df["demand_tur"] = np.maximum(df["demand_tur"].round().astype(int), 0)
    return df[["category", "demand_loc", "demand_tur"]]

In [11]:
def count_revenue(catalog, determine_df):
    df = determine_df.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
    df["revenue_loc"] = df["demand_loc"] * df["price_pln_base"]
    df["revenue_tur"] = df["demand_tur"] * df["price_pln_base"]
    df["revenue_pln"] = df["revenue_loc"] + df["revenue_tur"]
    return df[["category", "demand_loc", "demand_tur", "revenur_pln"]]

In [13]:
def generate_weekly_data(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    demand_df = determine_demand(catalog, rng, variance=variance, boost_loc=boost_loc, boost_tur=boost_tur)
    revenue_df = count_revenue(catalog, demand_df)
    return revenue_df

### Wygenerowanie danych tygodniowych

In [14]:
t1 = generate_weekly_data(catalog, rng)
t1 = t1.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
print("Tydzień 1 - przychody per kategoria (PLN)")
display(t1)
print("Łączny przychód w tygodniu 1 (PLN): ", t1["revenue_pln"].sum())

KeyError: "['demand_loc)base'] not in index"

# Analiza i wizualizacja danych

Podstawowe wykresy

In [16]:
t1.plot.bar()

NameError: name 't1' is not defined

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.barh(stacked=True, horizontal=True)

In [ ]:
hoights = t1["revenue_pln"]
plt.bar(range(len(hoights)), hoights)

Jawne tworzenie i wyświetlanie wykresu

In [ ]:
hoights = t1["revenue_pln"]
categories = t1["category"]

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(categories, hoights)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.show()

Etykiety wartości i formatowanie liczb